In [1]:
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

np.random.seed(42)
n = 300
time_idx = np.arange(n)
trend = time_idx * 0.05 
noise = np.random.normal(0, 1, n)
y = trend + noise

X = pd.DataFrame({"time_idx": time_idx})
y = pd.Series(y)

# Random KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = []
for train_idx, test_idx in kf.split(X):
    model = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    kf_scores.append(mean_absolute_error(y.iloc[test_idx], pred))

# TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
ts_scores = []
for train_idx, test_idx in tscv.split(X):
    model = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])
    ts_scores.append(mean_absolute_error(y.iloc[test_idx], pred))

print("KFold MAE:", np.mean(kf_scores), kf_scores)
print("TimeSeriesSplit MAE:", np.mean(ts_scores), ts_scores)

KFold MAE: 0.7782006469686742 [0.6315518648342486, 0.8435102790214636, 0.8189859446008723, 0.8423635879749577, 0.7545915584118288]
TimeSeriesSplit MAE: 0.847379259588837 [1.0208729937250711, 0.7960750524442105, 0.7134931954098209, 0.8435420557723657, 0.8629130005927166]


In [2]:
import duckdb
import pandas as pd

duckdb.sql("CREATE VIEW duolingo_flagship AS SELECT * FROM read_csv_auto('../../data/duolingo_flagship_v4.csv')")
df = duckdb.sql("SELECT * FROM duolingo_flagship").df()

df.shape

(16382, 17)

In [3]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits = 5)

In [4]:
for train_idx, test_idx in gkf.split(df, groups=df["user_id"]):
    print("train:", len(train_idx), "test:", len(test_idx))

train: 13105 test: 3277
train: 13105 test: 3277
train: 13106 test: 3276
train: 13106 test: 3276
train: 13106 test: 3276


In [6]:
for train_idx, test_idx in gkf.split(df, groups=df["user_id"]):
    train_users = set(df.iloc[train_idx]["user_id"])
    test_users = set(df.iloc[test_idx]["user_id"])
    overlap = train_users & test_users
    print("number of common users:", len(overlap))

number of common users: 0
number of common users: 0
number of common users: 0
number of common users: 0
number of common users: 0


In [10]:
import numpy as np

np.random.seed(42)
unique_users = df["user_id"].unique()
shuffled_idx = np.random.permutation(len(unique_users))
unique_users = np.array(unique_users)[shuffled_idx]

n_test_users = int(len(unique_users) * 0.15)
test_users = set(unique_users[:n_test_users])
cv_users = set(unique_users[n_test_users:])

df_test = df[df["user_id"].isin(test_users)]
df_cv = df[df["user_id"].isin(cv_users)]

print("hold-out test:", df_test.shape, "| CV pool:", df_cv.shape)
print("number of common users:", len(test_users & cv_users)) 

hold-out test: (2257, 17) | CV pool: (14125, 17)
number of common users: 0


## Split strategy

Grain: one row is one practice session for a (user, lexeme) pair. Sessions aren't independent,same user shows up in multiple rows.

Goal is predicting p_recall for a genuinely new user, not a known user's future session. That's the part that decides everything else.

Went with GroupKFold on user_id, no time component. Group structure is real (users repeat), so GroupKFold is needed. Checked this both in the mini lab and on the actual data, 0 overlapping users across all 5 folds. Didn't add a temporal split on top since the target scenario is "new user," not "future session" when a user shows up doesn't matter, only that their user_id never leaks across train and test.

Held out 15% of users as a final test set, untouched until evaluation (2,257 rows, ~338 users). The rest (14,125 rows, ~2,162 users) is the CV pool for GroupKFold during model development.